# LLM Model Testing Notebook

This notebook tests the three open-source medical LLM models from the S3 bucket:
1. **BioMistral-7B**
2. **meditron-7b**
3. **Meditron3-8B**

We'll check if all required files are present and test basic inference on each model.

## 1. Setup and Installation

In [1]:
# Install required packages
!pip install transformers torch accelerate sentencepiece -q

In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA A10G
CUDA memory: 22.06 GB


## 2. Define Model Paths

In [3]:
# Base path to models
BASE_PATH = "/home/ubuntu/nahiyan_workspace/s3_bucket/models"

# Model directories
MODELS = {
    "BioMistral-7B": os.path.join(BASE_PATH, "BioMistral-7B"),
    "meditron-7b": os.path.join(BASE_PATH, "meditron-7b"),
    "Meditron3-8B": os.path.join(BASE_PATH, "Meditron3-8B")
}

# Required files for each model
REQUIRED_FILES = [
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

print("Model paths configured:")
for name, path in MODELS.items():
    print(f"  {name}: {path}")

Model paths configured:
  BioMistral-7B: /home/ubuntu/nahiyan_workspace/s3_bucket/models/BioMistral-7B
  meditron-7b: /home/ubuntu/nahiyan_workspace/s3_bucket/models/meditron-7b
  Meditron3-8B: /home/ubuntu/nahiyan_workspace/s3_bucket/models/Meditron3-8B


## 3. Check Model Files Integrity

In [4]:
def check_model_files(model_name, model_path):
    """Check if all required files exist for a model"""
    print(f"\n{'='*60}")
    print(f"Checking {model_name}")
    print(f"{'='*60}")
    
    if not os.path.exists(model_path):
        print(f"❌ Model directory does not exist: {model_path}")
        return False
    
    print(f"✅ Model directory exists: {model_path}")
    
    # Check required files
    missing_files = []
    for file in REQUIRED_FILES:
        file_path = os.path.join(model_path, file)
        if os.path.exists(file_path):
            print(f"✅ {file}")
        else:
            print(f"❌ {file} - MISSING")
            missing_files.append(file)
    
    # Check for model weights (either .bin, .safetensors, or sharded)
    has_pytorch = os.path.exists(os.path.join(model_path, "pytorch_model.bin"))
    has_safetensors = os.path.exists(os.path.join(model_path, "model.safetensors"))
    has_sharded = os.path.exists(os.path.join(model_path, "pytorch_model.bin.index.json")) or \
                  os.path.exists(os.path.join(model_path, "model.safetensors.index.json"))
    
    if has_pytorch:
        print("✅ pytorch_model.bin (single file)")
    elif has_safetensors:
        print("✅ model.safetensors (single file)")
    elif has_sharded:
        print("✅ Sharded model weights detected")
        # List shard files
        shards = [f for f in os.listdir(model_path) if 'model-' in f or 'pytorch_model-' in f]
        print(f"   Found {len(shards)} shard files")
    else:
        print("❌ No model weight files found (.bin or .safetensors)")
        missing_files.append("model weights")
    
    # Check tokenizer model
    if os.path.exists(os.path.join(model_path, "tokenizer.model")):
        print("✅ tokenizer.model")
    else:
        print("⚠️  tokenizer.model - Not found (might not be required)")
    
    if missing_files:
        print(f"\n❌ Missing critical files: {', '.join(missing_files)}")
        return False
    else:
        print(f"\n✅ All required files present for {model_name}")
        return True

# Check all models
model_status = {}
for name, path in MODELS.items():
    model_status[name] = check_model_files(name, path)

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for name, status in model_status.items():
    status_icon = "✅" if status else "❌"
    print(f"{status_icon} {name}: {'Ready' if status else 'Missing files'}")


Checking BioMistral-7B
✅ Model directory exists: /home/ubuntu/nahiyan_workspace/s3_bucket/models/BioMistral-7B
✅ config.json
✅ tokenizer.json
✅ tokenizer_config.json
✅ pytorch_model.bin (single file)
✅ tokenizer.model

✅ All required files present for BioMistral-7B

Checking meditron-7b
✅ Model directory exists: /home/ubuntu/nahiyan_workspace/s3_bucket/models/meditron-7b
✅ config.json
✅ tokenizer.json
✅ tokenizer_config.json
✅ Sharded model weights detected
   Found 16 shard files
✅ tokenizer.model

✅ All required files present for meditron-7b

Checking Meditron3-8B
✅ Model directory exists: /home/ubuntu/nahiyan_workspace/s3_bucket/models/Meditron3-8B
✅ config.json
✅ tokenizer.json
✅ tokenizer_config.json
✅ Sharded model weights detected
   Found 4 shard files
⚠️  tokenizer.model - Not found (might not be required)

✅ All required files present for Meditron3-8B

SUMMARY
✅ BioMistral-7B: Ready
✅ meditron-7b: Ready
✅ Meditron3-8B: Ready


## 4. Test Model Loading and Inference

We'll test each model with a simple medical question.

In [5]:
# Test prompt - medical question
TEST_PROMPT = """Question: What are the common symptoms of Type 2 diabetes?
Answer:"""

def test_model_inference(model_name, model_path, max_length=150):
    """Load model and test inference"""
    print(f"\n{'='*60}")
    print(f"Testing {model_name}")
    print(f"{'='*60}")
    
    try:
        # Determine device
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {device}")
        
        # Load tokenizer
        print("Loading tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            local_files_only=True,
            trust_remote_code=True
        )
        print("✅ Tokenizer loaded successfully")
        
        # Load model
        print("Loading model... (this may take a few minutes)")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            local_files_only=True,
            trust_remote_code=True,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None,
            low_cpu_mem_usage=True
        )
        
        if device == "cpu":
            model = model.to(device)
        
        print("✅ Model loaded successfully")
        
        # Get model size
        param_count = sum(p.numel() for p in model.parameters())
        print(f"Model parameters: {param_count / 1e9:.2f}B")
        
        # Test inference
        print(f"\nRunning inference...")
        print(f"Prompt: {TEST_PROMPT}")
        
        inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        print(f"\n{'='*60}")
        print("Generated Output:")
        print(f"{'='*60}")
        print(generated_text)
        print(f"{'='*60}")
        
        # Clean up
        del model
        del tokenizer
        if device == "cuda":
            torch.cuda.empty_cache()
        
        print(f"\n✅ {model_name} inference test PASSED")
        return True
        
    except Exception as e:
        print(f"\n❌ {model_name} inference test FAILED")
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

### 4.1 Test BioMistral-7B

In [6]:
if model_status["BioMistral-7B"]:
    test_model_inference("BioMistral-7B", MODELS["BioMistral-7B"])
else:
    print("⚠️  Skipping BioMistral-7B - missing required files")


Testing BioMistral-7B
Using device: cuda
Loading tokenizer...
✅ Tokenizer loaded successfully
Loading model... (this may take a few minutes)


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded successfully
Model parameters: 7.24B

Running inference...
Prompt: Question: What are the common symptoms of Type 2 diabetes?
Answer:

Generated Output:
Question: What are the common symptoms of Type 2 diabetes?
Answer: Common symptoms of Type 2 diabetes include increased thirst and urination, increased hunger, weight loss, blurred vision, and slow healing of cuts and bruises.

✅ BioMistral-7B inference test PASSED


### 4.2 Test meditron-7b

In [7]:
if model_status["meditron-7b"]:
    test_model_inference("meditron-7b", MODELS["meditron-7b"])
else:
    print("⚠️  Skipping meditron-7b - missing required files")


Testing meditron-7b
Using device: cuda
Loading tokenizer...
✅ Tokenizer loaded successfully
Loading model... (this may take a few minutes)


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Model loaded successfully
Model parameters: 6.74B

Running inference...
Prompt: Question: What are the common symptoms of Type 2 diabetes?
Answer:

Generated Output:
Question: What are the common symptoms of Type 2 diabetes?
Answer: The most common symptoms of type 2 diabetes are polyuria, polydipsia, polyphagia, blurred vision, and fatigue.
Question: What is a typical blood sugar level for a person with diabetes?
Answer: In a person with diabetes, the fasting blood sugar is typically between 120 and 140 mg/dl.
Question: What is the purpose of checking blood sugar?
Answer: To check blood sugar, it is important to know what the blood sugar level is, the trend in the blood sugar level, and how to interpret the blood sugar level.
Question: How long does it take to become hypoglycem

✅ meditron-7b inference test PASSED


### 4.3 Test Meditron3-8B

In [8]:
if model_status["Meditron3-8B"]:
    test_model_inference("Meditron3-8B", MODELS["Meditron3-8B"])
else:
    print("⚠️  Skipping Meditron3-8B - missing required files")


Testing Meditron3-8B
Using device: cuda
Loading tokenizer...
✅ Tokenizer loaded successfully
Loading model... (this may take a few minutes)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully
Model parameters: 8.03B

Running inference...
Prompt: Question: What are the common symptoms of Type 2 diabetes?
Answer:

Generated Output:
Question: What are the common symptoms of Type 2 diabetes?
Answer: Common symptoms of type 2 diabetes include frequent urination, unusual thirst, and blurred vision. Other symptoms include fatigue, hunger, slow healing of cuts, tingling or numbness in hands and feet, and recurring skin, gum, or bladder infections.

✅ Meditron3-8B inference test PASSED


## 5. Final Summary

In [9]:
print("\n" + "="*60)
print("FINAL TEST SUMMARY")
print("="*60)
print("\nModel File Status:")
for name, status in model_status.items():
    status_icon = "✅" if status else "❌"
    print(f"{status_icon} {name}: {'All files present' if status else 'Missing files'}")

print("\n" + "="*60)
print("\nNext Steps:")
print("- If any models failed, check the error messages above")
print("- Ensure all model files are properly downloaded from Hugging Face")
print("- Check if you have sufficient GPU memory for inference")
print("- For CPU inference, expect slower performance")
print("\n" + "="*60)


FINAL TEST SUMMARY

Model File Status:
✅ BioMistral-7B: All files present
✅ meditron-7b: All files present
✅ Meditron3-8B: All files present


Next Steps:
- If any models failed, check the error messages above
- Ensure all model files are properly downloaded from Hugging Face
- Check if you have sufficient GPU memory for inference
- For CPU inference, expect slower performance



## 6. Additional Tests

You can add custom prompts below to test the models further.

In [11]:
# Custom prompt testing
CUSTOM_PROMPT = """Question: What is the recommended treatment for hypertension?
Answer:"""

# Uncomment to test with custom prompt
model_name = "BioMistral-7B"  # Change to test different models
if model_status[model_name]:
     test_model_inference(model_name, MODELS[model_name], max_length=200)


Testing BioMistral-7B
Using device: cuda
Loading tokenizer...
✅ Tokenizer loaded successfully
Loading model... (this may take a few minutes)
✅ Model loaded successfully
Model parameters: 7.24B

Running inference...
Prompt: Question: What are the common symptoms of Type 2 diabetes?
Answer:

Generated Output:
Question: What are the common symptoms of Type 2 diabetes?
Answer: Polyuria, polydipsia, polyphagia, weight loss, fatigue, blurred vision, slow healing, infections, atherosclerosis, heart disease, stroke, neuropathy, retinopathy, nephropathy, impotence in men, and depression.

✅ BioMistral-7B inference test PASSED
